In [1]:
import os
# Note: we use jax==0.4.38 + jax-metal==0.1.0 (the tested combo from darnax's pyproject.toml).
# jax-metal 0.1.x is NOT compatible with jax 0.9.x (default_memory_space not implemented).

import jax
import jax.numpy as jnp
import time

print(f"JAX version:  {jax.__version__}")
print(f"All devices:  {jax.devices()}")

# With jax-metal loaded, METAL becomes the default device.
# We also grab a CPU handle for comparison benchmarks.
cpu_dev = jax.devices("cpu")[0]

try:
    metal_dev = jax.devices("METAL")[0]
    metal_available = True
    print(f"\n✅ Metal GPU available: {metal_dev}")
except RuntimeError:
    metal_available = False
    metal_dev = None
    print("\n❌ Metal GPU NOT found (CPU only)")

print(f"   CPU: {cpu_dev}")

JAX version:  0.4.38
Metal device set to: Apple M3 Pro

systemMemory: 18.00 GB
maxCacheSize: 6.66 GB

All devices:  [METAL(id=0)]

✅ Metal GPU available: METAL:0
   CPU: TFRT_CPU_0


W0000 00:00:1774238155.375906 11182771 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1774238155.390400 11182771 service.cc:145] XLA service 0x1076814c0 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774238155.390535 11182771 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1774238155.392242 11182771 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1774238155.392292 11182771 mps_client.cc:384] XLA backend will use up to 14301773824 bytes on device 0 for SimpleAllocator.


In [2]:
# Verify we're using the right versions (run after kernel restart)
import jax
print(f"JAX version: {jax.__version__} (need 0.4.38)")
print(f"JAX location: {jax.__file__}")
assert jax.__version__ == "0.4.38", f"Wrong JAX version! Got {jax.__version__}. Please restart the kernel."
print("✅ Correct JAX version")

JAX version: 0.4.38 (need 0.4.38)
JAX location: /opt/anaconda3/envs/brainAI/lib/python3.11/site-packages/jax/__init__.py
✅ Correct JAX version


## Benchmark: CPU vs Metal GPU

We'll time key operations on both devices:
1. **Matrix multiplication** — the bread-and-butter GPU operation
2. **JIT-compiled matmul** — realistic, since JAX always JITs in practice
3. **Element-wise operations** — sign activations (like darnax uses)
4. **Convolution** — relevant for the conv architecture

For each, we warm up first (to exclude JIT compilation time), then time multiple runs.

In [3]:
import numpy as np

def benchmark(fn, n_warmup=3, n_runs=10, label=""):
    """Time a function after warmup. Returns median time in ms."""
    # Warmup (includes JIT compilation)
    for _ in range(n_warmup):
        result = fn()
        if hasattr(result, 'block_until_ready'):
            result.block_until_ready()
    
    # Timed runs
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        result = fn()
        if hasattr(result, 'block_until_ready'):
            result.block_until_ready()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)  # ms
    
    med = np.median(times)
    std = np.std(times)
    if label:
        print(f"  {label:30s}  {med:8.2f} ms  (±{std:.2f})")
    return med, std

print("Benchmark helper ready.")

Benchmark helper ready.


### 1. Matrix multiplication — varying sizes

In [4]:
sizes = [128, 256, 512, 1024, 2048, 4096]
matmul_results = {"size": [], "cpu_ms": [], "metal_ms": [], "speedup": []}

print("Matrix multiplication (A @ B), float32:")
print(f"  {'Size':>10s}  {'CPU':>10s}  {'Metal':>10s}  {'Speedup':>8s}")
print("  " + "-" * 46)

for n in sizes:
    # Generate data on CPU (the default device), then transfer
    key = jax.random.PRNGKey(0)
    a_data = jax.random.normal(key, (n, n))
    b_data = jax.random.normal(key, (n, n))
    
    # CPU
    a_cpu = jax.device_put(a_data, cpu_dev)
    b_cpu = jax.device_put(b_data, cpu_dev)
    cpu_ms, _ = benchmark(lambda: a_cpu @ b_cpu, label="")
    
    # Metal
    a_metal = jax.device_put(a_data, metal_dev)
    b_metal = jax.device_put(b_data, metal_dev)
    metal_ms, _ = benchmark(lambda: a_metal @ b_metal, label="")
    
    speedup = cpu_ms / metal_ms
    print(f"  {n:>10d}  {cpu_ms:>8.2f}ms  {metal_ms:>8.2f}ms  {speedup:>7.2f}×")
    
    matmul_results["size"].append(n)
    matmul_results["cpu_ms"].append(cpu_ms)
    matmul_results["metal_ms"].append(metal_ms)
    matmul_results["speedup"].append(speedup)

Matrix multiplication (A @ B), float32:
        Size         CPU       Metal   Speedup
  ----------------------------------------------
         128      0.03ms      0.52ms     0.07×
         256      0.20ms      0.24ms     0.82×
         512      0.61ms      0.49ms     1.24×
        1024      3.19ms      1.65ms     1.93×
        2048     21.44ms      3.53ms     6.07×
        4096    172.83ms     26.73ms     6.47×


### 2. JIT-compiled operations (realistic JAX usage)

In [5]:
# A JIT-compiled function combining matmul + activation (like one step in darnax)
@jax.jit
def forward_step(W, x):
    h = W @ x
    return jnp.sign(h)  # hard sign activation (like RecurrentDiscrete)

jit_results = {"size": [], "cpu_ms": [], "metal_ms": [], "speedup": []}

print("JIT-compiled: matmul + sign activation, float32:")
print(f"  {'Size':>10s}  {'CPU':>10s}  {'Metal':>10s}  {'Speedup':>8s}")
print("  " + "-" * 46)

for n in [256, 512, 1024, 2000, 4096]:
    key = jax.random.PRNGKey(42)
    W_data = jax.random.normal(key, (n, n))
    x_data = jnp.sign(jax.random.normal(key, (n,)))
    
    W_cpu = jax.device_put(W_data, cpu_dev)
    x_cpu = jax.device_put(x_data, cpu_dev)
    cpu_ms, _ = benchmark(lambda: forward_step(W_cpu, x_cpu), label="")
    
    W_metal = jax.device_put(W_data, metal_dev)
    x_metal = jax.device_put(x_data, metal_dev)
    metal_ms, _ = benchmark(lambda: forward_step(W_metal, x_metal), label="")
    
    speedup = cpu_ms / metal_ms
    print(f"  {n:>10d}  {cpu_ms:>8.2f}ms  {metal_ms:>8.2f}ms  {speedup:>7.2f}×")
    
    jit_results["size"].append(n)
    jit_results["cpu_ms"].append(cpu_ms)
    jit_results["metal_ms"].append(metal_ms)
    jit_results["speedup"].append(speedup)

JIT-compiled: matmul + sign activation, float32:
        Size         CPU       Metal   Speedup
  ----------------------------------------------
         256      0.01ms      0.52ms     0.03×
         512      0.03ms      0.23ms     0.13×
        1024      0.06ms      0.29ms     0.22×
        2000      0.26ms      0.67ms     0.38×
        4096      1.22ms      0.77ms     1.57×


### 3. 2D Convolution — darnax conv architecture workload

In [6]:
from jax import lax

@jax.jit
def conv2d_step(x, kernel):
    # x: (batch, H, W, C_in), kernel: (kH, kW, C_in, C_out)
    return lax.conv_general_dilated(
        x, kernel,
        window_strides=(1, 1),
        padding='SAME',
        dimension_numbers=('NHWC', 'HWIO', 'NHWC')
    )

conv_results = {"config": [], "cpu_ms": [], "metal_ms": [], "speedup": []}

configs = [
    # (batch, H, W, C_in, C_out, kernel_size)
    (32, 32, 32, 3, 64, 7),     # darnax input conv: CIFAR image → 64 channels
    (32, 32, 32, 64, 64, 7),    # darnax recurrent conv-like
    (64, 32, 32, 3, 64, 7),     # larger batch
    (32, 32, 32, 128, 128, 5),  # wider model
]

print("2D Convolution (NHWC, SAME padding), float32:")
print(f"  {'Config':>35s}  {'CPU':>10s}  {'Metal':>10s}  {'Speedup':>8s}")
print("  " + "-" * 70)

for (B, H, W, Ci, Co, K) in configs:
    key = jax.random.PRNGKey(0)
    k1, k2 = jax.random.split(key)
    
    # Generate on CPU, then transfer
    x_data = jax.random.normal(k1, (B, H, W, Ci))
    kernel_data = jax.random.normal(k2, (K, K, Ci, Co)) * 0.01
    
    x_cpu = jax.device_put(x_data, cpu_dev)
    k_cpu = jax.device_put(kernel_data, cpu_dev)
    cpu_ms, _ = benchmark(lambda: conv2d_step(x_cpu, k_cpu), label="")
    
    x_metal = jax.device_put(x_data, metal_dev)
    k_metal = jax.device_put(kernel_data, metal_dev)
    metal_ms, _ = benchmark(lambda: conv2d_step(x_metal, k_metal), label="")
    
    speedup = cpu_ms / metal_ms
    cfg_str = f"({B},{H},{W},{Ci})→{Co}, k={K}"
    print(f"  {cfg_str:>35s}  {cpu_ms:>8.2f}ms  {metal_ms:>8.2f}ms  {speedup:>7.2f}×")
    
    conv_results["config"].append(cfg_str)
    conv_results["cpu_ms"].append(cpu_ms)
    conv_results["metal_ms"].append(metal_ms)
    conv_results["speedup"].append(speedup)

2D Convolution (NHWC, SAME padding), float32:
                               Config         CPU       Metal   Speedup
  ----------------------------------------------------------------------
                 (32,32,32,3)→64, k=7      3.07ms      1.46ms     2.10×
                (32,32,32,64)→64, k=7     24.17ms      2.81ms     8.60×
                 (64,32,32,3)→64, k=7      5.32ms      1.84ms     2.90×
              (32,32,32,128)→128, k=5     42.83ms      5.30ms     8.08×


### 4. End-to-end: one darnax dynamics step (CPU vs Metal)

This is the real test — running the actual darnax `orchestrator.step()` on each device.

In [7]:
from darnax.modules.conv.conv import Conv2D, Conv2DRecurrentDiscrete
from darnax.modules.conv.pooling import GlobalUnpooling, GlobalMajorityPooling
from darnax.modules.fully_connected import FullyConnected, FrozenRescaledFullyConnected
from darnax.modules.recurrent import RecurrentDiscrete
from darnax.modules.input_output import OutputLayer
from darnax.layer_maps.sparse import LayerMap
from darnax.states.sequential import SequentialState
from darnax.orchestrators.sequential import SequentialOrchestrator
from darnax.trainers.dynamical import DynamicalTrainer
from darnax.trainers.utils import scan_n
import equinox as eqx
import optax
import jax.tree_util as jtu

N_CHANNELS = 64

def build_conv_model(key):
    """Build the same conv architecture used in notebook 9."""
    keys_net = jax.random.split(key, 5)
    
    input_to_hidden = Conv2D(
        in_channels=3, out_channels=N_CHANNELS, kernel_size=7,
        threshold=1.7, strength=1.0, key=keys_net[0], padding_mode='constant',
    )
    recurrent_conv = Conv2DRecurrentDiscrete(
        channels=N_CHANNELS, kernel_size=7, groups=N_CHANNELS,
        j_d=0.9, threshold=1.7, padding_mode="constant",
        key=keys_net[1], lr=1.0, weight_decay=0.0,
    )
    pooling_forward = GlobalMajorityPooling(strength=1.0, axis=(1, 2))
    unpooling_backward = GlobalUnpooling(strength=1.0, axis=(1, 2))
    recurrent = RecurrentDiscrete(
        features=N_CHANNELS, j_d=0.9, threshold=1.7, key=keys_net[2],
    )
    wout = FullyConnected(
        in_features=N_CHANNELS, out_features=10,
        strength=1.0, threshold=1.7, key=keys_net[3],
    )
    wback = FrozenRescaledFullyConnected(
        in_features=10, out_features=N_CHANNELS,
        strength=1.0, threshold=0.0, key=keys_net[4],
    )
    
    layer_map = LayerMap.from_dict({
        1: {0: input_to_hidden, 1: recurrent_conv, 2: unpooling_backward},
        2: {1: pooling_forward, 2: recurrent, 3: wback},
        3: {2: wout, 3: OutputLayer()},
    })
    return SequentialOrchestrator(layers=layer_map)


def run_dynamics_n_steps(orchestrator, state, rng, n_steps=11):
    """Run n dynamics steps (1 warmup + 5 clamped + 5 free)."""
    for _ in range(1):
        state, rng = orchestrator.step(state, rng=rng, filter_messages="forward")
    for _ in range(5):
        state, rng = orchestrator.step(state, rng=rng, filter_messages="all")
    for _ in range(5):
        state, rng = orchestrator.step(state, rng=rng, filter_messages="inference")
    return state, rng


# Build model + fake data (all generated on CPU)
key = jax.random.PRNGKey(0)
key, model_key, data_key = jax.random.split(key, 3)
orch = build_conv_model(model_key)

B = 32
x_data = jax.random.normal(data_key, (B, 32, 32, 3))
y_data = jnp.sign(jax.random.normal(data_key, (B, 10)))

# ---- CPU benchmark ----
orch_cpu = jax.device_put(orch, cpu_dev)
x_cpu = jax.device_put(x_data, cpu_dev)
y_cpu = jax.device_put(y_data, cpu_dev)
rng_cpu = jax.device_put(jax.random.PRNGKey(0), cpu_dev)

def run_cpu():
    s = SequentialState([(32,32,3), (32,32,N_CHANNELS), (N_CHANNELS,), (10,)])
    s = s.init(x_cpu, y_cpu)
    return run_dynamics_n_steps(orch_cpu, s, rng_cpu)

print("Benchmarking 11-step dynamics (1 warmup + 5 clamped + 5 free)...\n")

cpu_dyn, _ = benchmark(run_cpu, n_warmup=2, n_runs=5, label="CPU: 11 dynamics steps")

# ---- Metal benchmark ----
orch_metal = jax.device_put(orch, metal_dev)
x_met = jax.device_put(x_data, metal_dev)
y_met = jax.device_put(y_data, metal_dev)
rng_met = jax.device_put(jax.random.PRNGKey(0), metal_dev)

def run_metal():
    s = SequentialState([(32,32,3), (32,32,N_CHANNELS), (N_CHANNELS,), (10,)])
    s = s.init(x_met, y_met)
    return run_dynamics_n_steps(orch_metal, s, rng_met)

metal_dyn, _ = benchmark(run_metal, n_warmup=2, n_runs=5, label="Metal: 11 dynamics steps")

print(f"\n  Speedup: {cpu_dyn / metal_dyn:.2f}×")

Benchmarking 11-step dynamics (1 warmup + 5 clamped + 5 free)...

  CPU: 11 dynamics steps            630.02 ms  (±103.51)
  Metal: 11 dynamics steps          101.15 ms  (±5.48)

  Speedup: 6.23×


### 5. JIT-compiled full train step via `DynamicalTrainer` (CPU vs Metal)

This is the most realistic benchmark — it's exactly how the fast training loop works.

In [9]:
def build_trainer_on_device(device):
    """Build a DynamicalTrainer with all parameters on the given device."""
    key = jax.random.PRNGKey(0)
    key, model_key = jax.random.split(key)
    
    orch_dev = jax.device_put(build_conv_model(model_key), device)
    
    params, _ = eqx.partition(orch_dev, eqx.is_inexact_array)
    labels = jtu.tree_map(lambda _: "default", params, is_leaf=eqx.is_array)
    
    def like(tree, value):
        return jtu.tree_map(lambda _: value, tree, is_leaf=eqx.is_array)
    
    for (i, j), label in {(1,0): "w_in", (1,1): "j_conv", (2,2): "j_fc", (3,2): "w_out"}.items():
        labels = eqx.tree_at(
            lambda m, _i=i, _j=j: m.lmap[_i][_j], labels,
            replace=like(params.lmap[i][j], label),
        )
    
    optimizer = optax.multi_transform(
        {
            "default": optax.sgd(learning_rate=0.0),
            "w_in":    optax.sgd(learning_rate=0.1),
            "j_conv":  optax.sgd(learning_rate=0.05),
            "j_fc":    optax.sgd(learning_rate=0.05),
            "w_out":   optax.sgd(learning_rate=0.1),
        },
        labels,
    )
    
    state_template = SequentialState(
        [(32, 32, 3), (32, 32, N_CHANNELS), (N_CHANNELS,), (10,)]
    )
    opt_state = optimizer.init(eqx.filter(orch_dev, eqx.is_inexact_array))
    
    return DynamicalTrainer(
        orchestrator=orch_dev,
        state=state_template,
        optimizer=optimizer,
        optimizer_state=opt_state,
        warmup_n_iter=1,
        train_clamped_n_iter=5,
        train_free_n_iter=5,
        eval_n_iter=10,
    )


B = 32
key = jax.random.PRNGKey(99)
x_batch = jax.random.normal(key, (B, 32, 32, 3))
y_batch = jnp.sign(jax.random.normal(key, (B, 10)))
rng_key = jax.random.PRNGKey(0)

# ---- CPU ----
print("Building CPU trainer...")
trainer_cpu = build_trainer_on_device(cpu_dev)
x_cpu = jax.device_put(x_batch, cpu_dev)
y_cpu = jax.device_put(y_batch, cpu_dev)
rng_cpu = jax.device_put(rng_key, cpu_dev)

def train_step_cpu():
    return trainer_cpu.train_step(x_cpu, y_cpu, rng_cpu)

print("  Compiling JIT (CPU)... ", end="", flush=True)
t0 = time.perf_counter()
_ = train_step_cpu()
print(f"done in {time.perf_counter() - t0:.1f}s")
cpu_train, _ = benchmark(train_step_cpu, n_warmup=1, n_runs=5, label="CPU: full train step (JIT)")

# ---- Metal ----
print("\nBuilding Metal trainer...")
trainer_metal = build_trainer_on_device(metal_dev)
x_met = jax.device_put(x_batch, metal_dev)
y_met = jax.device_put(y_batch, metal_dev)
rng_met = jax.device_put(rng_key, metal_dev)

def train_step_metal():
    return trainer_metal.train_step(x_met, y_met, rng_met)

print("  Compiling JIT (Metal)... ", end="", flush=True)
t0 = time.perf_counter()
try:
    _ = train_step_metal()
    print(f"done in {time.perf_counter() - t0:.1f}s")
    metal_train, _ = benchmark(train_step_metal, n_warmup=1, n_runs=5, label="Metal: full train step (JIT)")
    print(f"\n  Full train step speedup: {cpu_train / metal_train:.2f}×")
except Exception as e:
    print(f"\n\n  ❌ Metal JIT compilation FAILED")
    print(f"  Error: {type(e).__name__}")
    # Extract the key info from the error
    err_msg = str(e)
    if "dot_general" in err_msg:
        print(f"\n  Root cause: the grouped depthwise convolution backward pass uses")
        print(f"  jnp.einsum('nhwklgc, nhwgo -> klgco', ...) which compiles to a")
        print(f"  batched dot_general that the Metal backend cannot legalize.")
        print(f"\n  This comes from Conv2DRecurrentDiscrete with groups={N_CHANNELS}")
        print(f"  in conv_backward_with_threshold() (conv/utils.py:402)")
    print(f"\n  ℹ️  The non-JIT dynamics step (benchmark 4) still works on Metal with 6.2× speedup!")
    print(f"  ℹ️  CPU JIT train step time: {cpu_train:.2f} ms")
    metal_train = None

Building CPU trainer...
  Compiling JIT (CPU)... done in 1.1s
  CPU: full train step (JIT)        831.50 ms  (±26.79)

Building Metal trainer...
  Compiling JIT (Metal)... 

  ❌ Metal JIT compilation FAILED
  Error: XlaRuntimeError

  Root cause: the grouped depthwise convolution backward pass uses
  jnp.einsum('nhwklgc, nhwgo -> klgco', ...) which compiles to a
  batched dot_general that the Metal backend cannot legalize.

  This comes from Conv2DRecurrentDiscrete with groups=64
  in conv_backward_with_threshold() (conv/utils.py:402)

  ℹ️  The non-JIT dynamics step (benchmark 4) still works on Metal with 6.2× speedup!
  ℹ️  CPU JIT train step time: 831.50 ms


## Summary & takeaways

### What works on Metal (Apple M3 Pro)
| Operation | Speedup vs CPU |
|---|---|
| Large matmul (≥1024) | 2–6.5× |
| 2D Convolution forward | 2–8.6× |
| Non-JIT dynamics (Python loop, 11 steps) | **6.2×** |
| JIT matmul + sign (≥4096) | 1.6× |

### What does NOT work on Metal
- **JIT-compiled `DynamicalTrainer.train_step()`** with the conv architecture — the grouped depthwise conv **backward** pass (`Conv2DRecurrentDiscrete` with `groups=N_CHANNELS`) uses a batched `einsum` that compiles to a `dot_general` which Metal cannot legalize.

### Practical implications
- **Forward dynamics** (inference, analysis) can use Metal for a ~6× speedup
- **Full training with JIT** must currently run on **CPU** (or the grouped einsum in `conv/utils.py` needs to be rewritten to avoid the unsupported HLO pattern)
- For **small tensors** (< 512), CPU is faster due to Metal dispatch overhead (~0.3 ms per call)

In [10]:
# ============================================================
# Quick-reference: how to use Metal in your notebooks
# ============================================================
# Copy this cell at the top of any notebook to enable Metal.
# Works with jax==0.4.38 + jax-metal==0.1.0.
# ============================================================

EXAMPLE_SETUP = """
import jax
import jax.numpy as jnp

# Detect devices
cpu  = jax.devices("cpu")[0]
try:
    metal = jax.devices("METAL")[0]
    USE_METAL = True
except RuntimeError:
    metal = None
    USE_METAL = False

# Choose target device for your workload
device = metal if USE_METAL else cpu

# Move model & data to device
orchestrator = jax.device_put(orchestrator, device)
x_batch      = jax.device_put(x_batch, device)
y_batch      = jax.device_put(y_batch, device)
rng          = jax.device_put(rng, device)

# Run dynamics (non-JIT) — works on Metal, ~6× faster
for i in range(n_steps):
    state, rng = orchestrator.step(state, rng=rng, ...)

# ⚠️ DynamicalTrainer.train_step() does NOT work on Metal
# with the conv architecture (grouped depthwise backward).
# Use device=cpu for JIT training, or use the non-JIT loop on Metal.
"""

print(EXAMPLE_SETUP)


import jax
import jax.numpy as jnp

# Detect devices
cpu  = jax.devices("cpu")[0]
try:
    metal = jax.devices("METAL")[0]
    USE_METAL = True
except RuntimeError:
    metal = None
    USE_METAL = False

# Choose target device for your workload
device = metal if USE_METAL else cpu

# Move model & data to device
orchestrator = jax.device_put(orchestrator, device)
x_batch      = jax.device_put(x_batch, device)
y_batch      = jax.device_put(y_batch, device)
rng          = jax.device_put(rng, device)

# Run dynamics (non-JIT) — works on Metal, ~6× faster
for i in range(n_steps):
    state, rng = orchestrator.step(state, rng=rng, ...)

# ⚠️ DynamicalTrainer.train_step() does NOT work on Metal
# with the conv architecture (grouped depthwise backward).
# Use device=cpu for JIT training, or use the non-JIT loop on Metal.

